# Monster hunting: relational anomaly exploration

This notebook explores the eight related tables **without fabricating results**. It loads tables one by one, validates the relationships shown in the data model, then searches for the six requested anomalies:

- **Phantom** — duplicate activity rows.
- **Ghost** — active in at least 3 of the first 4 Autumn weeks, then less than 10% of the early weekly activity on average.
- **Zombie** — continuous activity for more than 3 hours, with no gap longer than 15 minutes.
- **Vampire** — more activity days in the night (00:00–05:00) than in the day (05:00–00:00), and more than 21 night-activity days.
- **Undead student** — activity after deregistration.
- **Doppelgänger** — pairs of students with suspiciously identical activity signatures.
- **Shape-shifter** — evidence that a student switched programme/department during the year.

The notebook reports candidates and supporting evidence. It does not label a student as anomalous unless the relevant rule is met.

## 1. Configuration

Set `DATA_DIR` to the directory containing the exported table files. The loader supports CSV and Parquet. File names should contain the table names, for example:

- `la_mmh_person_dim.csv`
- `la_mmh_video_activity_by_view_fact.parquet`

If your files use different names, edit `TABLE_FILES` directly.

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

# Change this path before running.
DATA_DIR = Path("./data")

TABLES = {
    "calendar_dim": "la_mmh_calendar_dim",
    "person_dim": "la_mmh_person_dim",
    "module_instance_dim": "la_mmh_module_instance_dim",
    "video_dim": "la_mmh_video_dim",
    "lms_item_dim": "la_mmh_lms_item_dim",
    "person_module_instance_fact": "la_mmh_person_module_instance_fact",
    "video_activity_by_view_fact": "la_mmh_video_activity_by_view_fact",
    "lms_item_activity_by_session_fact": "la_mmh_lms_item_activity_by_session_fact",
}

# Optional explicit overrides. Leave values as None for automatic file discovery.
TABLE_FILES = {name: None for name in TABLES}

def _normalise_filename(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def find_table_file(table_key):
    explicit = TABLE_FILES.get(table_key)
    if explicit:
        path = Path(explicit)
        if not path.is_absolute():
            path = DATA_DIR / path
        if path.exists():
            return path
        raise FileNotFoundError(f"Configured file does not exist: {path}")

    expected = _normalise_filename(TABLES[table_key])
    candidates = []
    for pattern in ("*.csv", "*.CSV", "*.parquet", "*.pq"):
        candidates.extend(DATA_DIR.glob(pattern))

    matches = [p for p in candidates if expected in _normalise_filename(p.stem)]
    if not matches:
        raise FileNotFoundError(
            f"Could not find a CSV/Parquet file for {table_key!r} in {DATA_DIR.resolve()}. "
            f"Expected a name containing {TABLES[table_key]!r}."
        )
    return sorted(matches)[0]

def load_one_table(table_key):
    path = find_table_file(table_key)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")
    df.columns = [str(c).strip().upper() for c in df.columns]
    print(f"{table_key}: {path.name} | rows={len(df):,}, columns={len(df.columns)}")
    display(df.head(3))
    return df

## 2. Load and inspect every table, one at a time

In [ ]:
tables = {}
for table_key in TABLES:
    tables[table_key] = load_one_table(table_key)

The next cell creates standardised aliases for the two activity facts. The original columns remain available in `tables`; the aliases only make the anomaly rules robust to minor naming differences in exports.

In [ ]:
def first_existing(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found. Available columns: {list(df.columns)}")
    return None

def standardise_activity(df, source):
    out = df.copy()
    out["_SOURCE"] = source
    out["_P_ID"] = out[first_existing(out, ["P_ID"])]
    out["_TIMESTAMP"] = pd.to_datetime(
        out[first_existing(out, ["TIMESTAMP", "DATE"])],
        errors="coerce"
    )
    out["_DATE"] = pd.to_datetime(
        out[first_existing(out, ["DATE", "TIMESTAMP"])],
        errors="coerce"
    ).dt.normalize()
    return out

video = standardise_activity(tables["video_activity_by_view_fact"], "video")
lms = standardise_activity(tables["lms_item_activity_by_session_fact"], "lms")
activity = pd.concat([video, lms], ignore_index=True, sort=False)

print("Combined activity rows:", f"{len(activity):,}")
display(activity[["_SOURCE", "_P_ID", "_TIMESTAMP", "_DATE"]].head())

## 3. Basic quality checks and relationship validation

These checks are descriptive. A failed check is evidence about the exported data, not automatically one of the requested monsters.

In [ ]:
def key_report(df, key, label):
    if key not in df.columns:
        print(f"{label}: missing key column {key}")
        return
    nulls = df[key].isna().sum()
    dupes = df[key].duplicated(keep=False).sum()
    print(f"{label}: rows={len(df):,}; null {key}={nulls:,}; rows with duplicated {key}={dupes:,}")

for key, table_name, label in [
    ("P_ID", "person_dim", "person_dim"),
    ("MODULE_INSTANCE_ID", "module_instance_dim", "module_instance_dim"),
    ("VIDEO_ID", "video_dim", "video_dim"),
    ("LMS_ITEM_ID", "lms_item_dim", "lms_item_dim"),
]:
    key_report(tables[table_name], key, label)

def orphan_report(fact, fact_key, dim, dim_key, label):
    if fact_key not in fact.columns or dim_key not in dim.columns:
        print(f"{label}: skipped; key missing")
        return
    missing = fact[fact_key].notna() & ~fact[fact_key].isin(dim[dim_key].dropna().unique())
    print(f"{label}: orphan rows={missing.sum():,} ({missing.mean():.2%})")

orphan_report(video, "VIDEO_ID", tables["video_dim"], "VIDEO_ID", "video activity -> video_dim")
orphan_report(lms, "LMS_ITEM_ID", tables["lms_item_dim"], "LMS_ITEM_ID", "LMS activity -> lms_item_dim")
orphan_report(activity, "_P_ID", tables["person_dim"], "P_ID", "activity -> person_dim")
orphan_report(activity, "_P_ID", tables["person_module_instance_fact"], "P_ID", "activity -> person_module_instance_fact")

## 4. Phantom — duplicate activity rows

A row is treated as an exact duplicate when all identifying/activity columns available in that source are identical. Technical surrogate IDs are excluded where possible because a repeated event with a different surrogate ID is still potentially duplicated.

The result includes duplicate groups and their counts, so the finding can be audited.

In [ ]:
def duplicate_activity_groups(df, source):
    # Exclude technical IDs that may differ between otherwise identical event records.
    technical = {"PVT_ID", "LOG_IN_ID", "PMI_ID"}
    cols = [c for c in df.columns if c not in technical and not c.startswith("_")]
    cols = [c for c in cols if c in df.columns]
    dup_mask = df.duplicated(subset=cols, keep=False)
    dupes = df.loc[dup_mask].copy()
    if dupes.empty:
        return pd.DataFrame(), cols
    summary = (
        dupes.groupby(cols, dropna=False, sort=False)
        .size()
        .reset_index(name="DUPLICATE_COUNT")
        .sort_values("DUPLICATE_COUNT", ascending=False)
    )
    summary.insert(0, "SOURCE", source)
    return summary, cols

phantom_video, phantom_video_cols = duplicate_activity_groups(video, "video")
phantom_lms, phantom_lms_cols = duplicate_activity_groups(lms, "lms")
phantom = pd.concat([phantom_video, phantom_lms], ignore_index=True, sort=False)

print(f"Duplicate groups found: {len(phantom):,}")
if phantom.empty:
    print("Finding: no exact duplicate activity groups under this rule.")
else:
    print("Finding: these are candidate Phantom records; inspect whether repetition is expected.")
    display(phantom.head(20))

## 5. Ghost — early Autumn activity followed by a sharp decline

Rule implemented:

1. Identify the Autumn term from the calendar dimension.
2. Use the first four Autumn weeks.
3. A student must have activity in at least three of those four weeks.
4. Compute the mean weekly activity count during those early weeks.
5. Compute the mean weekly count over the remaining Autumn academic weeks.
6. Flag students whose later mean is below 10% of their early mean.

This uses activity-row counts from LMS and video together, as requested.

In [ ]:
cal = tables["calendar_dim"].copy()
for c in ["DATE", "YEAR", "MONTH", "DAY", "WEEK_NUMBER", "ACADEMIC_YEAR"]:
    if c in cal.columns:
        cal[c] = pd.to_datetime(cal[c], errors="coerce") if c == "DATE" else cal[c]

def autumn_mask(series):
    s = series.astype("string").str.upper()
    return s.str.contains("AUTUMN|FALL", regex=True, na=False)

term_col = "TERM" if "TERM" in cal.columns else None
if term_col is None:
    raise KeyError("calendar_dim needs TERM for the Ghost rule.")
autumn_cal = cal.loc[autumn_mask(cal[term_col])].copy()

if autumn_cal.empty:
    print("Finding: no Autumn rows detected in calendar_dim; Ghost analysis not run.")
else:
    week_col = first_existing(autumn_cal, ["WEEK_NUMBER"])
    autumn_weeks = sorted(pd.to_numeric(autumn_cal[week_col], errors="coerce").dropna().unique())
    print("Autumn weeks detected:", autumn_weeks)

    if len(autumn_weeks) < 4:
        print("Finding: fewer than four Autumn weeks detected; Ghost analysis not run.")
    else:
        first4 = autumn_weeks[:4]
        later = autumn_weeks[4:]

        date_to_week = autumn_cal.set_index("DATE")[week_col].to_dict() if "DATE" in autumn_cal else {}
        ghost_activity = activity.copy()
        ghost_activity["WEEK_NUMBER"] = ghost_activity["_DATE"].map(date_to_week)
        ghost_activity = ghost_activity[ghost_activity["WEEK_NUMBER"].isin(autumn_weeks)].copy()

        weekly = (
            ghost_activity.groupby(["_P_ID", "WEEK_NUMBER"])
            .size().rename("ACTIVITY_ROWS").reset_index()
        )
        students = weekly["_P_ID"].dropna().unique()
        base = pd.MultiIndex.from_product(
            [students, autumn_weeks], names=["_P_ID", "WEEK_NUMBER"]
        ).to_frame(index=False)
        weekly_full = base.merge(weekly, on=["_P_ID", "WEEK_NUMBER"], how="left").fillna({"ACTIVITY_ROWS": 0})

        early = weekly_full[weekly_full["WEEK_NUMBER"].isin(first4)].groupby("_P_ID").agg(
            EARLY_ACTIVE_WEEKS=("ACTIVITY_ROWS", lambda s: int((s > 0).sum())),
            EARLY_MEAN_WEEKLY_ACTIVITY=("ACTIVITY_ROWS", "mean"),
        )
        later_stats = weekly_full[weekly_full["WEEK_NUMBER"].isin(later)].groupby("_P_ID").agg(
            LATER_MEAN_WEEKLY_ACTIVITY=("ACTIVITY_ROWS", "mean"),
        )

        ghost = early.join(later_stats, how="inner").reset_index()
        ghost["LATER_AS_FRACTION_OF_EARLY"] = (
            ghost["LATER_MEAN_WEEKLY_ACTIVITY"] / ghost["EARLY_MEAN_WEEKLY_ACTIVITY"].replace(0, np.nan)
        )
        ghost = ghost[
            (ghost["EARLY_ACTIVE_WEEKS"] >= 3) &
            (ghost["LATER_AS_FRACTION_OF_EARLY"] < 0.10)
        ].sort_values(["EARLY_ACTIVE_WEEKS", "LATER_AS_FRACTION_OF_EARLY"], ascending=[False, True])

        print(f"Ghost candidates: {len(ghost):,}")
        display(ghost.head(50))

## 6. Zombie — uninterrupted activity over more than three hours

All LMS and video timestamps are merged per student. Consecutive events are considered part of the same run when the gap is at most 15 minutes. A candidate run must span **strictly more than three hours** from its first to last timestamp.

This is a timestamp-based screen, not proof that the student was continuously engaged between events.

In [ ]:
z = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
z = z.sort_values(["_P_ID", "_TIMESTAMP"])
z["GAP_MINUTES"] = z.groupby("_P_ID")["_TIMESTAMP"].diff().dt.total_seconds().div(60)
z["RUN_ID"] = z["GAP_MINUTES"].gt(15).groupby(z["_P_ID"]).cumsum()

zombie_runs = (
    z.groupby(["_P_ID", "RUN_ID"], as_index=False)
     .agg(
         START=("_TIMESTAMP", "min"),
         END=("_TIMESTAMP", "max"),
         EVENTS=("_TIMESTAMP", "size"),
         MAX_GAP_MINUTES=("GAP_MINUTES", "max"),
     )
)
zombie_runs["DURATION_HOURS"] = (
    zombie_runs["END"] - zombie_runs["START"]
).dt.total_seconds() / 3600

zombie = zombie_runs[zombie_runs["DURATION_HOURS"] > 3].sort_values(
    ["DURATION_HOURS", "EVENTS"], ascending=False
)
print(f"Zombie runs: {len(zombie):,}; students: {zombie['_P_ID'].nunique() if not zombie.empty else 0:,}")
display(zombie.head(50))

## 7. Vampire — more active days at night than during the day

Night is defined as 00:00–05:00. Day is defined as 05:00–00:00. Dates with at least one activity event are counted, not raw event counts.

The requested threshold is enforced: more than 21 night-activity days and more night days than day days.

In [ ]:
v = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
v["HOUR"] = v["_TIMESTAMP"].dt.hour
v["PERIOD"] = np.where(v["HOUR"] < 5, "NIGHT", "DAY")

v_days = v.groupby(["_P_ID", "PERIOD"])["_DATE"].nunique().unstack(fill_value=0)
for c in ["NIGHT", "DAY"]:
    if c not in v_days:
        v_days[c] = 0

vampire = v_days.reset_index().rename(columns={
    "NIGHT": "NIGHT_ACTIVITY_DAYS",
    "DAY": "DAY_ACTIVITY_DAYS",
})
vampire = vampire[
    (vampire["NIGHT_ACTIVITY_DAYS"] > 21) &
    (vampire["NIGHT_ACTIVITY_DAYS"] > vampire["DAY_ACTIVITY_DAYS"])
].sort_values(["NIGHT_ACTIVITY_DAYS", "DAY_ACTIVITY_DAYS"], ascending=False)

print(f"Vampire candidates: {len(vampire):,}")
display(vampire.head(50))

## 8. Undead student — activity after deregistration

A student is flagged when an activity timestamp occurs strictly after their `DEREGISTRATION_DATE`. The report is split by source and includes the first and last post-deregistration activity.

In [ ]:
people = tables["person_dim"].copy()
people["DEREGISTRATION_DATE"] = pd.to_datetime(people["DEREGISTRATION_DATE"], errors="coerce")
undead = activity.merge(
    people[["P_ID", "DEREGISTRATION_DATE"]],
    left_on="_P_ID", right_on="P_ID", how="left"
)
undead = undead[
    undead["DEREGISTRATION_DATE"].notna() &
    undead["_TIMESTAMP"].notna() &
    (undead["_TIMESTAMP"].dt.normalize() > undead["DEREGISTRATION_DATE"])
]

undead_summary = (
    undead.groupby(["_P_ID", "DEREGISTRATION_DATE"], as_index=False)
    .agg(
        SOURCES=("_SOURCE", lambda s: ", ".join(sorted(s.unique()))),
        POST_DEREG_ACTIVITY_ROWS=("_TIMESTAMP", "size"),
        FIRST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "min"),
        LAST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "max"),
    )
    .sort_values("POST_DEREG_ACTIVITY_ROWS", ascending=False)
)
print(f"Undead students: {len(undead_summary):,}")
display(undead_summary.head(50))

## 9. Doppelgänger — suspiciously identical activity

There is no single canonical definition of “identical activity” in the schema, so this notebook makes the rule explicit and auditable:

- create a per-student signature from both activity facts;
- count rows by source, calendar date, hour, and available resource ID;
- compare students with the same complete signature.

Students are only reported when their signatures match exactly. This is intentionally conservative; a looser similarity model can be added later.

In [ ]:
d = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
d["DATE_KEY"] = d["_TIMESTAMP"].dt.date.astype("string")
d["HOUR_KEY"] = d["_TIMESTAMP"].dt.hour

resource_cols = [c for c in ["VIDEO_ID", "LMS_ITEM_ID"] if c in d.columns]
d["RESOURCE_KEY"] = pd.NA
for c in resource_cols:
    d["RESOURCE_KEY"] = d["RESOURCE_KEY"].fillna(d[c].astype("string"))

signature_detail = (
    d.groupby(["_P_ID", "_SOURCE", "DATE_KEY", "HOUR_KEY", "RESOURCE_KEY"], dropna=False)
     .size().rename("N").reset_index()
)
signature = (
    signature_detail.groupby("_P_ID", sort=False)
    .apply(lambda g: tuple(map(tuple, g.drop(columns="_P_ID").sort_values(
        ["_SOURCE", "DATE_KEY", "HOUR_KEY", "RESOURCE_KEY"]
    ).to_numpy())))
    .rename("SIGNATURE").reset_index()
)

groups = signature.groupby("SIGNATURE", dropna=False)["_P_ID"].agg(list).reset_index()
groups["PAIR_COUNT"] = groups["_P_ID"].str.len()
doppel_groups = groups[groups["PAIR_COUNT"] >= 2].copy()

pairs = []
for _, row in doppel_groups.iterrows():
    ids = sorted(map(str, row["_P_ID"]))
    for i, a in enumerate(ids):
        for b in ids[i+1:]:
            pairs.append({"P_ID_A": a, "P_ID_B": b, "MATCHED_SIGNATURE_ROWS": row["PAIR_COUNT"]})
doppelganger = pd.DataFrame(pairs)

print(f"Doppelgänger pairs: {len(doppelganger):,}")
if doppelganger.empty:
    print("Finding: no exact signature matches under this conservative rule.")
else:
    display(doppelganger.head(50))

## 10. Shape-shifter — programme/department changes

The supplied model does not contain a field literally named `PROGRAMME`. The closest available programme-like field is `DEPARTMENT_ID` in `module_instance_dim`.

This screen therefore flags students linked through `person_module_instance_fact` to more than one department across module instances. It also reports the academic year and term where available. Treat this as a candidate list for review, not a definitive programme-transfer label.

In [ ]:
pm = tables["person_module_instance_fact"].copy()
mi = tables["module_instance_dim"].copy()

shape = pm.merge(
    mi[["MODULE_INSTANCE_ID", "ACADEMIC_YEAR_START", "TERM", "DEPARTMENT_ID"]],
    on="MODULE_INSTANCE_ID", how="left"
)

dept_counts = (
    shape.dropna(subset=["P_ID", "DEPARTMENT_ID"])
    .groupby("P_ID")
    .agg(
        DEPARTMENT_COUNT=("DEPARTMENT_ID", "nunique"),
        DEPARTMENTS=("DEPARTMENT_ID", lambda s: ", ".join(sorted(map(str, s.dropna().unique())))),
        MODULE_INSTANCE_COUNT=("MODULE_INSTANCE_ID", "nunique"),
    )
    .reset_index()
)
shape_shifter = dept_counts[dept_counts["DEPARTMENT_COUNT"] > 1].sort_values(
    ["DEPARTMENT_COUNT", "MODULE_INSTANCE_COUNT"], ascending=False
)

print(f"Shape-shifter candidates: {len(shape_shifter):,}")
display(shape_shifter.head(50))

if not shape_shifter.empty:
    print("Supporting module history for the first candidates:")
    display(
        shape[shape["P_ID"].isin(shape_shifter["P_ID"].head(10))]
        [["P_ID", "MODULE_INSTANCE_ID", "ACADEMIC_YEAR_START", "TERM", "DEPARTMENT_ID"]]
        .sort_values(["P_ID", "ACADEMIC_YEAR_START", "TERM"])
    )

## 11. Final candidate counts and export

The export contains candidate IDs and supporting measures. Empty result sets are valid findings: they mean the rule did not identify any records in the supplied data.

The notebook deliberately does not manufacture a final “monster label”. These candidate tables are suitable inputs for later feature engineering and predictive modelling.

In [ ]:
summary = pd.DataFrame({
    "MONSTER": [
        "Phantom", "Ghost", "Zombie", "Vampire",
        "Undead student", "Doppelgänger", "Shape-shifter"
    ],
    "CANDIDATES_OR_GROUPS": [
        len(phantom),
        len(ghost) if "ghost" in globals() else 0,
        len(zombie),
        len(vampire),
        len(undead_summary),
        len(doppelganger),
        len(shape_shifter),
    ],
})
display(summary)

OUTPUT_DIR = Path("./monster_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save only objects that were created successfully.
exports = {
    "phantom.csv": phantom,
    "ghost.csv": ghost if "ghost" in globals() else pd.DataFrame(),
    "zombie_runs.csv": zombie,
    "vampire.csv": vampire,
    "undead_student.csv": undead_summary,
    "doppelganger_pairs.csv": doppelganger,
    "shape_shifter.csv": shape_shifter,
    "candidate_summary.csv": summary,
}
for filename, df in exports.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)

print(f"Saved {len(exports)} result files to {OUTPUT_DIR.resolve()}")

## Interpretation and next modelling step

Before training a predictive model:

1. Review candidate rows with domain owners; some anomalies may be legitimate.
2. Preserve the rule outputs as explainable labels/features, not as unquestionable truth.
3. Add features such as activity intensity, active-day counts, inter-event gaps, post-registration timing, resource diversity, and module/department transitions.
4. Split training and validation by student or by time to avoid leakage.
5. Keep the exact thresholds and assumptions in the model documentation.

No conclusions about actual students are made until the notebook is run against the real table files.